In [83]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sksurv.util import Surv
from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis, ComponentwiseGradientBoostingSurvivalAnalysis
from sksurv.metrics import cumulative_dynamic_auc
from sklearn.metrics import classification_report, confusion_matrix

import catboost as cb

In [2]:
df = pd.read_csv("../data/tr_complete.csv")
df

,vehicle_id,time_step,171_0,666_0,427_0,837_0,167_0,167_1,167_2,167_3,...,is_Spec_0,is_Spec_1,is_Spec_2,is_Spec_3,is_Spec_4,is_Spec_5,is_Spec_6,is_Spec_7,length_of_study_time_step,in_study_repair
0,0,11.2,167985.0,10787.0,7413813.0,2296.0,4110.0,1296420.0,1628265.0,630345.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,510.0,0
1,0,11.4,167985.0,10787.0,7413813.0,2296.0,4111.0,1302855.0,1628265.0,630345.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,510.0,0
2,0,19.6,331635.0,14525.0,13683604.0,2600.0,4111.0,1302855.0,1628265.0,630345.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,510.0,0
3,0,20.2,354975.0,15015.0,14540449.0,2616.0,4111.0,1302855.0,1628265.0,630345.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,510.0,0
4,0,21.0,365550.0,15295.0,14966985.0,2720.0,4111.0,1302855.0,1628265.0,630345.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,510.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1122447,33643,101.0,2136810.0,41412.0,81068660.0,10365.0,843.0,6408272.0,10964376.0,6092347.0,...,0.0,0.0,12.0,0.0,0.0,0.0,1.0,4.0,130.2,0
1122448,33643,107.0,2283285.0,44660.0,86639830.0,12317.0,843.0,6596477.0,12358026.0,6721702.0,...,0.0,0.0,12.0,0.0,0.0,0.0,1.0,4.0,130.2,0
1122449,33643,113.8,2462865.0,48370.0,93439416.0,13910.0,843.0,6669542.0,12830421.0,7415497.0,...,0.0,0.0,12.0,0.0,0.0,0.0,1.0,4.0,130.2,0
1122450,33643,119.8,2627250.0,51101.0,99682930.0,14854.0,843.0,7074468.0,13604706.0,7747492.0,...,0.0,0.0,12.0,0.0,0.0,0.0,1.0,4.0,130.2,0


In [3]:
EDGES = np.array([6, 12, 24, 48])
COST = np.array([[0, 7, 8, 9, 10],
                 [200, 0, 7, 8, 9],
                 [300, 200, 0, 7, 8],
                 [400, 300, 200, 0, 7],
                 [500, 400, 300, 200, 0]])
CUT_FRACS = [.3, .5, .7, .9, .95, .98, 1.]

In [4]:
def make_snapshots(df: pd.DataFrame, fracs):
    rows = []
    n_len = df["vehicle_id"].nunique()
    print(n_len)
    n = 0
    for vid, g in df.sort_values("time_step").groupby("vehicle_id"):
        n += 1
        if n % 500 == 0:
            print(n)
        t = g["time_step"].to_numpy()
        for f in fracs:
            cut = np.percentile(t, f * 100)
            rows.append(g[g["time_step"] <= cut].iloc[-1])
    snap = pd.DataFrame(rows).drop_duplicates(["vehicle_id", "time_step"])
    snap["tte"] = (snap["length_of_study_time_step"] - snap["time_step"]).clip(lower=1e-3)
    return snap.reset_index(drop=True)

In [5]:
def predict_classes(model, X):
    predict = model.predict_survival_function(X)
    S = np.array([[s(t) for t in EDGES] for s in predict])
    P = np.c_[S[:, 3], S[:, 2]-S[:, 3], S[:, 1]-S[:, 2], S[:, 0]-S[:, 1], 1-S[:, 0]]
    return np.argmin(P @ COST, axis=1)

In [6]:
def true_class(tte, event):
    cls = 4 - np.searchsorted(EDGES, tte, side="left")
    return np.where(~event & (tte <= EDGES[-1]), -1, cls)


In [7]:
snap = make_snapshots(df, CUT_FRACS)

23550
500
1000
1500
2000
2500
3000
3500
4000
4500
5000
5500
6000
6500
7000
7500
8000
8500
9000
9500
10000
10500
11000
11500
12000
12500
13000
13500
14000
14500
15000
15500
16000
16500
17000
17500
18000
18500
19000
19500
20000
20500
21000
21500
22000
22500
23000
23500


In [8]:
snap.to_csv("snap.csv", index=False)

In [9]:
del df
del snap

In [10]:
snap = pd.read_csv("snap.csv")

In [11]:
snap

,vehicle_id,time_step,171_0,666_0,427_0,837_0,167_0,167_1,167_2,167_3,...,is_Spec_1,is_Spec_2,is_Spec_3,is_Spec_4,is_Spec_5,is_Spec_6,is_Spec_7,length_of_study_time_step,in_study_repair,tte
0,0.0,138.4,3090960.0,123583.0,118443780.0,24931.0,-1.0,-1.0,-1.0,-1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,510.0,0.0,371.6
1,0.0,271.8,5560740.0,206809.0,-1.0,30896.0,-1.0,-1.0,-1.0,-1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,510.0,0.0,238.2
2,0.0,353.0,7158600.0,265836.0,-1.0,35092.0,-1.0,-1.0,-1.0,-1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,510.0,0.0,157.0
3,0.0,462.4,9358410.0,341066.0,-1.0,39153.0,-1.0,-1.0,-1.0,-1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,510.0,0.0,47.6
4,0.0,473.4,9595755.0,350327.0,-1.0,39596.0,-1.0,-1.0,-1.0,-1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,510.0,0.0,36.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159383,33643.0,82.2,1731300.0,32219.0,65903316.0,7684.0,841.0,5128756.0,8734701.0,4913886.0,...,0.0,12.0,0.0,0.0,0.0,1.0,4.0,130.2,0.0,48.0
159384,33643.0,107.0,2283285.0,44660.0,86639830.0,12317.0,843.0,6596477.0,12358026.0,6721702.0,...,0.0,12.0,0.0,0.0,0.0,1.0,4.0,130.2,0.0,23.2
159385,33643.0,113.8,2462865.0,48370.0,93439416.0,13910.0,843.0,6669542.0,12830421.0,7415497.0,...,0.0,12.0,0.0,0.0,0.0,1.0,4.0,130.2,0.0,16.4
159386,33643.0,119.8,2627250.0,51101.0,99682930.0,14854.0,843.0,7074468.0,13604706.0,7747492.0,...,0.0,12.0,0.0,0.0,0.0,1.0,4.0,130.2,0.0,10.4


In [12]:
snap["tte"] = np.maximum(np.round(snap["tte"]), 1)

In [13]:
X = snap.drop(columns=["length_of_study_time_step", "vehicle_id", "tte", "in_study_repair"]).fillna(-1)

In [14]:
event = snap["in_study_repair"].astype(bool).to_numpy()
y = Surv.from_arrays(event=event, time=snap["tte"].to_numpy())

In [15]:
groups = snap["vehicle_id"].to_numpy()

# Случайный лес

In [16]:
oof = np.zeros(len(snap), dtype=int)
auc_s = []
for tr, va in StratifiedGroupKFold(5, shuffle=True, random_state=42).split(X, event, groups):
    model = RandomSurvivalForest(n_estimators=10, random_state=42, min_samples_leaf=10, max_features="sqrt", n_jobs=-1, verbose=1, max_samples=0.3)
    model.fit(X.iloc[tr], y[tr])
    oof[va] = predict_classes(model, X.iloc[va])

    ev_times = y[va]["time"][y[va]["event"]]
    times = np.linspace(*np.percentile(ev_times, [10, 90]), 20)

    auc, mean_auc = cumulative_dynamic_auc(y[tr], y[va], model.predict(X.iloc[va]), times)
    auc_s.append(mean_auc)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 out of  10 | elapsed:  2.6min finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   2 out of  10 | elapsed:    0.5s remaining:    1.8s
[Parallel(n_jobs=10)]: Done  10 out of  10 | elapsed:    0.6s finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   2 out of  10 | elapsed:    0.5s remaining:    2.0s
[Parallel(n_jobs=10)]: Done  10 out of  10 | elapsed:    0.6s finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 out of  10 | elapsed:  2.4min finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   2 out of  10 | elapsed:    0.3s remaining:    1.0s
[Parallel(n_jobs=10)]: Done  10 out of  10 | elapsed:    0.4s f

In [17]:
tc = true_class(snap["tte"].to_numpy(), event)
ok = tc >= 0
print("model cost:", COST[tc[ok], oof[ok]].sum())
print("const baselines:", [COST[tc[ok], k].sum() for k in range(5)])

model cost: 886151
const baselines: [np.int64(3458400), np.int64(2803900), np.int64(1911007), np.int64(1307316), np.int64(746641)]


In [18]:
np.mean(auc_s)

np.float64(0.7008774753982315)

In [19]:
print(classification_report(tc[ok], oof[ok], zero_division=0.0))

              precision    recall  f1-score   support

           0       0.96      0.44      0.61     69700
           1       0.02      0.01      0.01      2001
           2       0.02      0.02      0.02      2344
           3       0.02      0.04      0.02      1840
           4       0.07      0.85      0.13      3238

    accuracy                           0.43     79123
   macro avg       0.22      0.27      0.16     79123
weighted avg       0.85      0.43      0.54     79123



In [20]:
confusion_matrix(tc[ok], oof[ok])

array([[30809,   777,  2226,  4020, 31868],
       [  350,    20,    54,    88,  1489],
       [  307,    11,    42,   113,  1871],
       [  227,     9,    40,    78,  1486],
       [  313,    13,    57,   114,  2741]])

In [21]:
pd.Series(COST[tc[ok], oof[ok]]).groupby(tc[ok]).sum()

0    378107
1     84483
2    110059
3    111902
4    201600
dtype: int64

In [39]:
m = RandomSurvivalForest(n_estimators=10, random_state=42, min_samples_leaf=10, max_features="sqrt", n_jobs=-1, verbose=2, max_samples=0.3)
m.fit(X, y)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 16 concurrent workers.


building tree 1 of 10
building tree 2 of 10
building tree 3 of 10
building tree 4 of 10
building tree 5 of 10
building tree 6 of 10
building tree 7 of 10
building tree 8 of 10
building tree 9 of 10
building tree 10 of 10


[Parallel(n_jobs=-1)]: Done   3 out of  10 | elapsed:  4.6min remaining: 10.8min
[Parallel(n_jobs=-1)]: Done  10 out of  10 | elapsed:  5.5min finished


NotImplementedError: 

In [44]:
import shap
import numpy as np
import pandas as pd

X_background = shap.sample(X, 100, random_state=42)
X_explain = shap.sample(X, 200, random_state=42)

def predict_risk(X):
    return m.predict(X)

max_evals = 2 * X.shape[1] + 1

explainer = shap.Explainer(
    predict_risk,
    X_background,
    algorithm="permutation",
    max_evals=max_evals
)

shap_values = explainer(X_explain)

[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   3 out of  10 | elapsed:    0.7s remaining:    1.6s
[Parallel(n_jobs=10)]: Done  10 out of  10 | elapsed:    0.7s finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   3 out of  10 | elapsed:    0.7s remaining:    1.7s
[Parallel(n_jobs=10)]: Done  10 out of  10 | elapsed:    0.8s finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   3 out of  10 | elapsed:    0.3s remaining:    0.6s
[Parallel(n_jobs=10)]: Done  10 out of  10 | elapsed:    0.3s finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   3 out of  10 | elapsed:    0.6s remaining:    1.4s
[Parallel(n_jobs=10)]: Done  10 out of  10 | elapsed:    0.7s finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 co

In [46]:
importance = np.abs(shap_values.values).mean(axis=0)

feature_importance = (
    pd.DataFrame({
        "feature": X.columns,
        "importance": importance
    })
    .sort_values("importance", ascending=False)
)

feature_importance

,feature,importance
15,309_0,4.993440
74,397_4,4.894838
44,158_5,4.615336
73,397_3,4.555214
294,459_14_percent,4.295882
...,...,...
208,has_397_21,0.000000
209,has_397_22,0.000000
210,has_397_23,0.000000
211,has_397_24,0.000000


In [48]:
y_pred = predict_classes(m, X)

[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   3 out of  10 | elapsed:    1.8s remaining:    4.2s
[Parallel(n_jobs=10)]: Done  10 out of  10 | elapsed:    2.7s finished


In [51]:
y_pred[ok]

array([3, 4, 4, ..., 0, 0, 2], shape=(79123,))

In [50]:
tc[ok]

array([0, 0, 0, ..., 0, 0, 0], shape=(79123,))

In [54]:
print(classification_report(tc[ok], y_pred[ok]))

              precision    recall  f1-score   support

           0       1.00      0.47      0.64     69700
           1       0.03      0.01      0.02      2001
           2       0.01      0.01      0.01      2344
           3       0.01      0.02      0.01      1840
           4       0.08      1.00      0.15      3238

    accuracy                           0.46     79123
   macro avg       0.22      0.30      0.17     79123
weighted avg       0.88      0.46      0.57     79123



In [58]:
p = m.predict_survival_function(X)


[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   3 out of  10 | elapsed:    1.5s remaining:    3.5s
[Parallel(n_jobs=10)]: Done  10 out of  10 | elapsed:    2.4s finished


In [65]:
p[0](7)

np.float64(0.9800000000000001)

In [70]:
tc[0]

np.int64(0)

In [77]:
for t in [6, 12, 24, 48]:
    print(p[0](t))

1.0
0.9727272727272727
0.9727272727272727
0.9383522727272726


In [81]:
for i, j in zip(*np.unique_counts([col[:3] for col in snap.columns])):
    print(i, j)

100 4
158 21
167 21
171 3
272 21
291 23
309 4
370 4
397 73
427 4
459 41
666 4
835 4
837 4
has 104
in_ 1
is_ 8
len 1
tim 1
tte 1
veh 1


# Градиентный бустинг sksurv

In [32]:
oof = np.zeros(len(snap), dtype=int)
auc_s = []
for tr, va in StratifiedGroupKFold(5, shuffle=True, random_state=42).split(X, event, groups):
    model = ComponentwiseGradientBoostingSurvivalAnalysis(
        n_estimators=5,
        random_state=42,
        learning_rate=0.1,
        # max_depth=2,
        verbose=1,
        # max_features="sqrt",
        # subsample=0.5,
    )
    model.fit(X.iloc[tr], y[tr])
    oof[va] = predict_classes(model, X.iloc[va])

    ev_times = y[va]["time"][y[va]["event"]]
    times = np.linspace(*np.percentile(ev_times, [10, 90]), 20)

    auc, mean_auc = cumulative_dynamic_auc(y[tr], y[va], model.predict(X.iloc[va]), times)
    auc_s.append(mean_auc)

      Iter       Train Loss   Remaining Time 
         1      135070.8618            5.62m
         2      135063.1095            4.16m
         3      135055.4637            2.78m
         4      135047.8599            1.35m
         5      135040.2984            0.00s
      Iter       Train Loss   Remaining Time 
         1      134982.1090            5.49m
         2      134974.2204            4.13m
         3      134966.4391            2.77m
         4      134958.7647            1.38m
         5      134951.1966            0.00s
      Iter       Train Loss   Remaining Time 
         1      135135.6659            5.48m
         2      135127.2266            4.11m
         3      135118.9029            2.73m
         4      135110.6576            1.33m
         5      135102.4761            0.00s
      Iter       Train Loss   Remaining Time 
         1      135024.0094            5.51m
         2      135015.0544            4.16m
         3      135006.2220            2.77m
      

In [33]:
tc = true_class(snap["tte"].to_numpy(), event)
ok = tc >= 0
print("model cost:", COST[tc[ok], oof[ok]].sum())
print("const baselines:", [COST[tc[ok], k].sum() for k in range(5)])

model cost: 746641
const baselines: [np.int64(3458400), np.int64(2803900), np.int64(1911007), np.int64(1307316), np.int64(746641)]


In [34]:
np.mean(auc_s)

np.float64(0.6286931944645721)

In [35]:
print(classification_report(tc[ok], oof[ok], zero_division=0.0))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00     69700
           1       0.00      0.00      0.00      2001
           2       0.00      0.00      0.00      2344
           3       0.00      0.00      0.00      1840
           4       0.04      1.00      0.08      3238

    accuracy                           0.04     79123
   macro avg       0.01      0.20      0.02     79123
weighted avg       0.00      0.04      0.00     79123



In [36]:
confusion_matrix(tc[ok], oof[ok])

array([[    0,     0,     0,     0, 69700],
       [    0,     0,     0,     0,  2001],
       [    0,     0,     0,     0,  2344],
       [    0,     0,     0,     0,  1840],
       [    0,     0,     0,     0,  3238]])

In [37]:
pd.Series(COST[tc[ok], oof[ok]]).groupby(tc[ok]).sum()

0    697000
1     18009
2     18752
3     12880
4         0
dtype: int64

# CatBoost

In [84]:
model = cb.CatBoostClassifier(
    loss_function="MultiClass",
    learning_rate=0.5,
    depth=5,
    thread_count=-1,
    random_seed=42,
    verbose=True,
    iterations=100,
)

In [86]:
model.fit(X[ok], tc[ok])

0:	learn: 0.6256315	total: 105ms	remaining: 10.3s
1:	learn: 0.5397262	total: 146ms	remaining: 7.15s
2:	learn: 0.4985934	total: 190ms	remaining: 6.13s
3:	learn: 0.4791251	total: 229ms	remaining: 5.49s
4:	learn: 0.4683286	total: 267ms	remaining: 5.07s
5:	learn: 0.4611828	total: 311ms	remaining: 4.88s
6:	learn: 0.4569069	total: 361ms	remaining: 4.79s
7:	learn: 0.4529216	total: 414ms	remaining: 4.76s
8:	learn: 0.4491088	total: 458ms	remaining: 4.63s
9:	learn: 0.4466874	total: 499ms	remaining: 4.49s
10:	learn: 0.4456245	total: 537ms	remaining: 4.34s
11:	learn: 0.4445158	total: 578ms	remaining: 4.24s
12:	learn: 0.4432896	total: 615ms	remaining: 4.11s
13:	learn: 0.4411840	total: 651ms	remaining: 4s
14:	learn: 0.4394715	total: 692ms	remaining: 3.92s
15:	learn: 0.4384867	total: 730ms	remaining: 3.83s
16:	learn: 0.4372205	total: 772ms	remaining: 3.77s
17:	learn: 0.4366369	total: 810ms	remaining: 3.69s
18:	learn: 0.4358092	total: 849ms	remaining: 3.62s
19:	learn: 0.4345688	total: 891ms	remaining:

CatBoostClassifier(depth=5, iterations=100, learning_rate=0.5, loss_function='MultiClass', random_seed=42, verbose=True)

In [87]:
y_pred = model.predict(X[ok])

In [89]:
print(classification_report(tc[ok], y_pred))

              precision    recall  f1-score   support

           0       0.90      1.00      0.95     69700
           1       0.71      0.01      0.02      2001
           2       0.56      0.06      0.11      2344
           3       0.63      0.04      0.07      1840
           4       0.58      0.23      0.33      3238

    accuracy                           0.89     79123
   macro avg       0.68      0.27      0.30     79123
weighted avg       0.86      0.89      0.85     79123



In [90]:
confusion_matrix(tc[ok], y_pred)

array([[69587,     2,    11,     1,    99],
       [ 1878,    25,    24,     6,    68],
       [ 2013,     6,   140,     9,   176],
       [ 1535,     1,    32,    66,   206],
       [ 2413,     1,    45,    22,   757]])